In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList, LogitsProcessor

# Doing it the long way with generate()

In [ ]:
class ConstrainedLogitsProcessor(LogitsProcessor):
    def __init__(self, allowed_tokens):
        self.allowed_tokens = allowed_tokens
        self.applied = False

    def __call__(self, input_ids, logits): # HF LogitsProcessor needs input_ids to run
        if self.applied:
            return logits
        
        mask = torch.full(logits.shape, float('-inf')).to(logits.device) # essentially some tokens are impossible
        mask[:, self.allowed_tokens] = 0  # Only allow specific tokens for each one generated
        logits = logits + mask
        self.applied = True

        return logits

In [50]:
MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Encode input text
input_text = "This is a test"
input_ids = tokenizer.encode(input_text, return_tensors="pt")  # Tokenized input

# Get logits
with torch.no_grad():
    outputs = model(input_ids=input_ids)
    logits = outputs.logits  # Raw logits output

# Print logits for the next token prediction
print("Logits shape:", logits.shape)  # Should be [batch_size, seq_length, vocab_size]

Logits shape: torch.Size([1, 4, 50257])


In [68]:
allowed_labels=['r']
allowed_tokens = [tokenizer.convert_tokens_to_ids(label) for label in allowed_labels]

logits_processor = LogitsProcessorList([ConstrainedLogitsProcessor(allowed_tokens)])

input_text = "Return a list of digits containing 9 only: "
input_ids = tokenizer.encode(input_text, return_tensors="pt")  # Tokenized input

output = model.generate(
    input_ids=input_ids,
    max_new_tokens=10,
    logits_processor=logits_processor,
    return_dict_in_generate=True,
    output_scores=True,
)

print("Generated tokens:", output.sequences) 
print("Generated logits:", torch.softmax(output.scores[0][0][:25], dim=-1))  # Only len(allowed_labels) tokens in each sequence

# Decode generated tokens
print(tokenizer.decode(output.sequences[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated tokens: tensor([[13615,   257,  1351,   286, 19561,  7268,   860,   691,    25,   220,
            81,   796,   352,    26,   329,   357,    72,   796,   657,    26]])
Generated logits: tensor([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan])
Return a list of digits containing 9 only: r = 1; for (i = 0;
